QuickBuy is an online retailer that wants to improve its customer shopping experience. Customers can leave reviews and ratings for products they have purchased.  

QuickBuy needs to keep track of its data using both a relational database (SQLite) and a NoSQL database (MongoDB) and object-oriented programming techniques.  

The following shows the relational schema for the SQLite database `quickbuy.db` with three tables:
- `customers (customer_id, customer_name, email)`
- `products (product_id, product_name, category)`
- `orders (customer_id#, product_id#, order_date, quantity)`
`# denotes foreign key`

**Task 3.1**  
Write program code to create the three tables for the relational database. (6)

- correct setup workflow
- CREATE TABLE IF NOT EXISTS
- correct field name and appropriate field type (2)
- correct primary key (3)
- correct foreign key (2)

In [4]:
#Task 3.1
import sqlite3

conn = sqlite3.connect('quickbuy.db')
cur = conn.cursor()

cur.execute('''CREATE TABLE IF NOT EXISTS customers(
    customer_id TEXT PRIMARY KEY,
    customer_name TEXT,
    email TEXT);''')

cur.execute('''CREATE TABLE IF NOT EXISTS products(
    product_id TEXT PRIMARY KEY,
    product_name TEXT,
    category TEXT);''')

cur.execute('''CREATE TABLE IF NOT EXISTS orders(
    customer_id TEXT,
    product_id TEXT,
    order_date TEXT,
    quantity INTEGER,
    PRIMARY KEY(customer_id, product_id, order_date),
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id));''')

conn.commit()
conn.close()

**Task 3.2**  
Write program code to insert records from customers.txt, products.txt and orders.txt into the three tables. (5)

- correct file input processing (2)
- INSERT INTO with placeholder (3)

In [6]:
# Task 3.2
conn = sqlite3.connect('quickbuy.db')
cur = conn.cursor()

with open('customers.txt', 'r') as file:
    lines = file.readlines()
    for line in lines:
        line = line.strip().split(',')
        cur.execute("INSERT INTO customers(customer_id, customer_name, email) VALUES (?, ?, ?)" \
                    , (line[0], line[1], line[2]))
    # file automatically closed

with open('products.txt', 'r') as file:
    lines = file.readlines()
    for line in lines:
        line = line.strip().split(',')
        cur.execute("INSERT INTO products(product_id, product_name, category) VALUES (?, ?, ?)" \
                    , (line[0], line[1], line[2]))
    # file automatically closed

with open('orders.txt', 'r') as file:
    lines = file.readlines()
    for line in lines:
        line = line.strip().split(',')
        cur.execute("INSERT INTO orders(customer_id, product_id, order_date, quantity) VALUES (?, ?, ?, ?)", \
                    (line[0], line[1], line[2], line[3]))
    # file automatically closed

conn.commit()
conn.close()

**Task 3.3**  
Write program code to get and output the total quantity of products ordered by each customer. (3)

- correct SELECT statement with necessary fields, aggregate function and group by (3)
- correct output

In [7]:
# Task 3.3
conn = sqlite3.connect('quickbuy.db')
cur = conn.cursor()

cur.execute("SELECT customer_id, SUM(quantity) FROM orders GROUP BY customer_id;")
recs = cur.fetchall()
print(recs)

conn.close()

[('C1', 4), ('C10', 3), ('C2', 3), ('C3', 3), ('C4', 3), ('C5', 3), ('C6', 4), ('C7', 4), ('C8', 3), ('C9', 3)]


**Task 3.4**  
Write program code to get and output the number of orders and total quantity for each product category for orders made in the year 2023. (4)

- correct SELECT statement with aggregate function and criteria (4)
- correct (empty) output (edge case)  
Note: for testing, can change year to 2024 to confirm all records are returned

In [17]:
# Task 3.4
conn = sqlite3.connect('quickbuy.db')
cur = conn.cursor()

cur.execute('''SELECT p.category, COUNT(*) AS num_orders, SUM(o.quantity) AS total_qty
FROM products p, orders o 
WHERE p.product_id = o.product_id AND o.order_date LIKE '2023%'
GROUP BY p.category;''')
recs = cur.fetchall()
print(recs)

conn.close()

[]


**Task 3.5**  
Write program code to insert data from reviews.json to a quickbuy MongoDB database reviews collection. (3)

- correct setup workflow
- correct file input processing
- appropriate data structure
- correct insert logic

In [20]:
# Task 3.5
import json
from pymongo import MongoClient

with open("reviews.json", "r") as file:
    reviews = json.load(file)
# file automatically closed

# connect MongoDB
client = MongoClient('localhost', 27017)
db = client['quickbuy']
coll = db['reviews']

# insert reviews
result = coll.insert_many(reviews)

# output status
print(f"Inserted {len(result.inserted_ids)} documents.")

# close connection
client.close()

Inserted 6 documents.


**Task 3.6**
Write program code to find and output all highly rated products (at least rating 4) with qualitative reviews. (3)

- correct query logic (rating, review text, find) (3)
- correct output

In [22]:
# Task 3.6
# connect MongoDB
client = MongoClient('localhost', 27017)
db = client['quickbuy']
coll = db['reviews']

query = {"rating": {"$gte": 4}, "review_text": {"$exists": True, "$ne": ""}}
docs = coll.find(query)

print("Highly rated products with qualitative reviews:")
for doc in docs:
    print(f"\nProduct: {doc['product_name']}")
    print(f"Product ID: {doc['product_id']}")
    print(f"Rating: {doc['rating']}")
    print(f"Review: {doc['review_text']}")
    print(f"Review Date: {doc['review_date']}")
    print(f"Customer ID: {doc['customer_id']}")

# close connection
client.close()

Highly rated products with qualitative reviews:

Product: Ultra HD Smart TV
Product ID: P5
Rating: 5
Review: Amazing picture quality and smart features! The colors are vibrant and the interface is user-friendly.
Review Date: 2024-08-28
Customer ID: C1

Product: Ergonomic Office Chair
Product ID: P3
Rating: 4
Review: Very comfortable for long working hours. The lumbar support is great, but I wish the armrests were a bit more adjustable.
Review Date: 2024-08-27
Customer ID: C2


**Task 3.7**
Write program code to find and output all the reviews for product P5 and calculate and output its average rating correct to 1 decimal place. (4)

- correct filter logic (3)
- correct compute average logic 
- correct output (1 decimal place)

In [27]:
# Task 3.7
# connect MongoDB
client = MongoClient('localhost', 27017)
db = client['quickbuy']
coll = db['reviews']

# query product P5
query = {"product_id": "P5"}
docs = coll.find(query)

# store ratings for average calculation
ratings = []

# output reviews and collect ratings
print("Reviews:")
for doc in docs:
    print(f"\nCustomer ID: {doc['customer_id']}")
    print(f"Rating: {doc['rating']}")
    if 'review_text' in doc and doc['review_text']:
        print(f"Review: {doc['review_text']}")
    print(f"Review Date: {doc['review_date']}")

    ratings.append(doc['rating'])

# calculate and output average rating
if ratings:
    ave_rating = sum(ratings) / len(ratings)
    print(f"\nAverage rating: {round(ave_rating, 1)}")
else:
    print("\nNo ratings found")

# close connection
client.close()

Reviews:

Customer ID: C1
Rating: 5
Review: Amazing picture quality and smart features! The colors are vibrant and the interface is user-friendly.
Review Date: 2024-08-28

Customer ID: C6
Rating: 4
Review Date: 2024-08-23

Average rating: 4.5
